In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def fetch_megaselah_soups():
    """Fetches the latest Megaselah records from Catagolue."""
    # Catagolue's text samples endpoint for megaselahs in standard Life (b3s23)
    url = "https://catagolue.hatsya.com/text/samples/b3s23/megaselah"
    
    response = requests.get(url)
    response.raise_for_status()
    
    lines = response.text.strip().split('\n')
    
    records = []
    # Skip the header row and parse the CSV-style text payload
    for line in lines[1:]: 
        parts = line.split(',')
        if len(parts) >= 2:
            records.append({
                "apgcode": parts[0].strip(' "'),
                "soup_string": parts[1].strip(' "')
            })
            
    return pd.DataFrame(records)

def plot_soup(soup_string):
    """
    Decodes a standard apgsearch soup string into a 16x16 matrix 
    and plots it. (Assumes a 64-character hex or base32 encoding).
    """
    # Catagolue soups are often prefixed with 'm_' or similar
    clean_string = soup_string.split('_')[-1] 
    
    # Convert string to binary, pad to 256 bits, and reshape to 16x16
    try:
        # Note: Exact decoding depends on the specific apgsearch version encoding.
        # This handles standard hex representations.
        binary_string = bin(int(clean_string, 16))[2:].zfill(256)
        matrix = np.array([int(bit) for bit in binary_string]).reshape(16, 16)
        
        plt.figure(figsize=(4, 4))
        plt.imshow(matrix, cmap='binary')
        plt.title(f"Soup: {soup_string[:12]}...")
        plt.axis('off')
        plt.show()
    except ValueError:
        print(f"Could not decode soup string: {soup_string}")

# 1. Load the API data into a DataFrame
df = fetch_megaselah_soups()

# 2. Display the top records
display(df.head())

# 3. Visualize the starting seed of the first Megaselah found
if not df.empty:
    first_soup = df.iloc[0]['soup_string']
    plot_soup(first_soup)

HTTPError: 404 Client Error: Not Found for url: https://catagolue.hatsya.com/text/samples/b3s23/megaselah